In [1]:
import pandas as pd
import mysql.connector
from datetime import datetime

In [55]:
import os

config = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_NAME', 'hotel_bookings')
}

if not config['user'] or not config['password']:
    raise ValueError("Please configure DB_USER and DB_PASSWORD.")

conn = mysql.connector.connect(**config)
cursor = conn.cursor()


In [75]:
raw_data = pd.read_csv('hotel_bookings.csv', nrows=20)


In [77]:
raw_data.at[5, 'arrival_date_year'] = 2025


In [79]:
# 1. Total nights 
raw_data['total_nights'] = raw_data['stays_in_weekend_nights'] + raw_data['stays_in_week_nights']

# 2. Month number from month name 
raw_data['arrival_date_month_num'] = pd.to_datetime(raw_data['arrival_date_month'], format='%B').dt.month

# 3. Fill missing agent and company with 0
raw_data['agent'] = raw_data['agent'].fillna(0).astype(int)
raw_data['company'] = raw_data['company'].fillna(0).astype(int)

# 4. Create unified arrival_date
raw_data['arrival_date'] = pd.to_datetime(
    raw_data['arrival_date_year'].astype(str) + '-' +
    raw_data['arrival_date_month'] + '-' +
    raw_data['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d'
)

# 5. Total revenue (only for non-canceled bookings)
raw_data['revenue'] = raw_data.apply(
    lambda row: row['adr'] * row['total_nights'] if row['is_canceled'] == 0 else 0, axis=1
)

# 6. Ensure reservation_status_date is in datetime format
raw_data['reservation_status_date'] = pd.to_datetime(raw_data['reservation_status_date'])

# 7. Add weekday number (Monday=1, Sunday=7) and weekday name from arrival_date
raw_data['weekday_num'] = raw_data['arrival_date'].dt.weekday + 1
raw_data['weekday_name'] = raw_data['arrival_date'].dt.strftime('%A')


In [87]:
raw_data.head(10)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,total_nights,arrival_date_month_num,arrival_date,revenue,weekday_num,weekday_name
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0,0,Check-Out,2015-07-01,0,7,2015-07-01,0.0,3,Wednesday
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,0,0,Check-Out,2015-07-01,0,7,2015-07-01,0.0,3,Wednesday
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,0,Check-Out,2015-07-02,1,7,2015-07-01,75.0,3,Wednesday
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,0,Check-Out,2015-07-02,1,7,2015-07-01,75.0,3,Wednesday
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,1,Check-Out,2015-07-03,2,7,2015-07-01,196.0,3,Wednesday
5,Resort Hotel,0,14,2025,July,27,1,0,2,2,...,0,1,Check-Out,2015-07-03,2,7,2025-07-01,196.0,2,Tuesday
6,Resort Hotel,0,0,2015,July,27,1,0,2,2,...,0,0,Check-Out,2015-07-03,2,7,2015-07-01,214.0,3,Wednesday
7,Resort Hotel,0,9,2015,July,27,1,0,2,2,...,0,1,Check-Out,2015-07-03,2,7,2015-07-01,206.0,3,Wednesday
8,Resort Hotel,1,85,2015,July,27,1,0,3,2,...,0,1,Canceled,2015-05-06,3,7,2015-07-01,0.0,3,Wednesday
9,Resort Hotel,1,75,2015,July,27,1,0,3,2,...,0,0,Canceled,2015-04-22,3,7,2015-07-01,0.0,3,Wednesday


In [83]:
conn = mysql.connector.connect(**config)
cursor = conn.cursor()

def load_dim_table(table_name, id_col, name_col):
    cursor.execute(f"SELECT {id_col}, {name_col} FROM {table_name}")
    rows = cursor.fetchall()
    return dict(rows)

# Load mappings for string-based dimensions
hotel_map = load_dim_table('dim_hotel', 'hotel_id', 'hotel')
meal_map = load_dim_table('dim_meal', 'meal_id', 'meal')
country_map = load_dim_table('dim_country', 'country_id', 'country')
market_segment_map = load_dim_table('dim_market_segment', 'market_segment_id', 'market_segment')
distribution_channel_map = load_dim_table('dim_distribution_channel', 'distribution_channel_id', 'distribution_channel')
customer_type_map = load_dim_table('dim_customer_type', 'customer_type_id', 'customer_type')
deposit_type_map = load_dim_table('dim_deposit_type', 'deposit_type_id', 'deposit_type')
reservation_status_map = load_dim_table('dim_reservation_status', 'reservation_status_id', 'reservation_status')
room_type_map = load_dim_table('dim_room_type', 'room_type_id', 'room_type')

# Load mapping for dim_date (datetime mapping)
cursor.execute("SELECT date_id, date FROM dim_date")
date_rows = cursor.fetchall()
date_map = {row[1]: row[0] for row in date_rows}


In [97]:
conn = mysql.connector.connect(**config)
cursor = conn.cursor()

# Mapping: dimension table name -> (column in raw_data, column in dim_table)
dim_mappings = {
    'dim_date' : ('arrival_date', 'date'),
    'dim_hotel': ('hotel', 'hotel'),
    'dim_meal': ('meal', 'meal'),
    'dim_country': ('country', 'country'),
    'dim_market_segment': ('market_segment', 'market_segment'),
    'dim_distribution_channel': ('distribution_channel', 'distribution_channel'),
    'dim_customer_type': ('customer_type', 'customer_type'),
    'dim_deposit_type': ('deposit_type', 'deposit_type'),
    'dim_reservation_status': ('reservation_status', 'reservation_status'),
    'dim_room_type': ('reserved_room_type', 'room_type')  # reserved & assigned share this dim
}

for table, (raw_col, dim_col) in dim_mappings.items():
    # Get unique values from raw_data
    raw_values = set(raw_data[raw_col].dropna().unique())
    
    # Get existing values from dimension table
    cursor.execute(f"SELECT {dim_col} FROM {table}")
    existing = set(row[0] for row in cursor.fetchall())

    # Find missing values
    to_insert = raw_values - existing
    if to_insert:
        print(f"Inserting {len(to_insert)} new values into {table}: {to_insert}")
        for value in to_insert:
            cursor.execute(f"INSERT INTO {table} ({dim_col}) VALUES (%s)", (value,))
        conn.commit()
    else:
        print(f"No new values to insert into {table}")

cursor.close()
conn.close()
print("Dimension tables updated.")


Inserting 1 new values into dim_date: {Timestamp('2025-07-01 00:00:00')}


DatabaseError: 1364 (HY000): Field 'date_id' doesn't have a default value

In [91]:
print(raw_values)

{'E', 'G', 'C', 'D', 'A'}


In [59]:
insert_fact = """
INSERT INTO fact_bookings (
    hotel_id, arrival_date_id, reservation_status_date_id, meal_id, country_id,
    market_segment_id, distribution_channel_id, customer_type_id, deposit_type_id,
    reservation_status_id, reserved_room_type_id, assigned_room_type_id, is_canceled,
    lead_time, stays_in_weekend_nights, stays_in_week_nights, adults, children, babies,
    is_repeated_guest, previous_cancellations, previous_bookings_not_canceled,
    booking_changes, agent, company, days_in_waiting_list, adr, required_car_parking_spaces,
    total_of_special_requests, total_nights, revenue
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

fact_columns = [
    'hotel_id', 'arrival_date_id', 'reservation_status_date_id', 'meal_id', 'country_id',
    'market_segment_id', 'distribution_channel_id', 'customer_type_id', 'deposit_type_id',
    'reservation_status_id', 'reserved_room_type_id', 'assigned_room_type_id', 'is_canceled',
    'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies',
    'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled',
    'booking_changes', 'agent', 'company', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces',
    'total_of_special_requests', 'total_nights', 'revenue'
]


In [63]:
conn = mysql.connector.connect(**config)
cursor = conn.cursor()

batch_size = 10
values = []

for idx, row in raw_data.iterrows():
    # convert NaN to None for SQL
    row_values = [row[col] if pd.notna(row[col]) else None for col in fact_columns]
    values.append(tuple(row_values))

    if len(values) >= batch_size:
        cursor.executemany(insert_fact, values)
        conn.commit()
        print(f"Inserted {len(values)} rows up to index {idx}")
        values = []

# Insert any remaining rows
if values:

    cursor.executemany(insert_fact, values)
    conn.commit()
    print(f"Inserted last {len(values)} rows")

cursor.close()
conn.close()
print("ETL process completed!")


Inserted 10 rows up to index 9
Inserted 10 rows up to index 19
ETL process completed!
